# Neighborhoods with Most Tree Canopy Change

Which neighborhoods lose tree canopy the fastest?

Are there hotspots of tree canopy loss or gain?

In [11]:
import rasterio
import geopandas as gpd
import pandas as pd
import numpy as np
import re
from rasterio.mask import mask
from shapely.geometry import mapping

In [2]:
lcdf = pd.read_csv('lulc_visualization_crosswalk-2024-edition.csv')

In [3]:
#change rasters found at: https://www.sciencebase.gov/catalog/item/68011c83d4be0263cab101c8

change1to2 = "../landcover_spring2026_toobig/lcc/balt_24510_lulc-change_2013-2018_2024-Edition.tif"
change1to3 = "../landcover_spring2026_toobig/lcc/balt_24510_lulc-change_2013-2021_2024-Edition.tif"                      
change2to3 = "../landcover_spring2026_toobig/lcc/balt_24510_lulc-change_2018-2021_2024-Edition.tif"


In [4]:
#land cover raster

balt_2013 = "../landcover_spring2026_toobig/lc/balt_24510_lulc_2013_2024-Edition.tif"

In [5]:
def landcover_polygon(
    raster_path,
    lookup_df,
    polygons_gdf,
    #output_csv,
    pixel_col="pixel_value",
    class_col="class_name",
    neighborhood_name_col="Name"
):

    #use geodataframe with neighborhood_id
    gdf = polygons_gdf.copy()

    if "neighborhood_id" not in gdf.columns:
        raise ValueError("GeoDataFrame must contain 'neighborhood_id'")

    # Build pixel value → class dictionary
    lookup_df = lookup_df.copy()

    

    value_to_class = dict(zip(lookup_df[pixel_col], lookup_df[class_col]))

    results = []

    with rasterio.open(raster_path) as src:

        # Ensure CRS match
        if gdf.crs != src.crs:
            gdf = gdf.to_crs(src.crs)

        for idx, row in gdf.iterrows():

            geom = [mapping(row.geometry)]

            out_image, _ = mask(src, geom, crop=True)
            data = out_image[0]
            # mask(src, geom, crop=True)[0]

            # Remove nodata
            data = data[data != src.nodata]

            if len(data) == 0:
                continue

            # Convert pixel values to class names
            classes = pd.Series(data).map(value_to_class)

            # Count pixels per class
            class_counts = classes.value_counts()

            # Convert to acres
            
            class_acres = class_counts / 4046.8564224

            total_acres = class_acres.sum()

            row_dict = {
                "neighborhood_id": row["neighborhood_id"],
                neighborhood_name_col: row[neighborhood_name_col]
            }

            #print(row["neighborhood_id"])
            #print(row[neighborhood_name_col])

            total_percent = 0

            for lc_class, acres in class_acres.items():
                
                percent = (acres / total_acres) * 100 if total_acres > 0 else 0
    
                row_dict[f"{lc_class}_acres".replace(" ","_")] = acres
                row_dict[f"{lc_class}_percent".replace(" ","_")] = percent
    
                total_percent += percent

            results.append(row_dict)

    results_df = pd.DataFrame(results).fillna(0)
    results_df["total_percent"] = total_percent
    final_df = results_df.copy()


    return final_df


In [6]:
#lookup table function


lcdf = pd.read_csv('lulc_visualization_crosswalk-2024-edition.csv') 


def merge_classes(lookup_df, class_col="class_name"):
    
    lookup_df = lookup_df.copy()
    
    lookup_df[class_col] = lookup_df[class_col].str.lower()
    
    def merge_name(name):
        if "tree canopy" in name:
            return "tree canopy"
        elif "impervious" in name:
            return "impervious"
        else:
            return name
    
    lookup_df[class_col] = lookup_df[class_col].apply(merge_name)
    
    return lookup_df

crosswalk_lc = merge_classes(lcdf, class_col="lc") 
pd.reset_option('display.max_rows')
crosswalk_lc

,value,lulc,lu,lc,general,macro
0,10,Tidal Waters,Tidal Waters,water,Water,Water
1,11,Lakes and Reservoirs,Lakes and Reservoirs,water,Water,Water
2,12,Riverine Ponds,Riverine Ponds,water,Water,Water
3,13,Terrene Ponds,Terrene Ponds,water,Water,Water
4,14,Streams and Rivers,Streams and Rivers,water,Water,Water
...,...,...,...,...,...,...
57,91,Animal Operation Impervious,Animal Operation,impervious,"Impervious, Other",Agricultural
58,92,Animal Operation Barren,Animal Operation,barren,Pasture and Hay,Agricultural
59,93,Animal Operation Herbaceous,Animal Operation,low vegetation,Pasture and Hay,Agricultural
60,94,Tree Canopy over Agricultural Structure,Animal Operation,tree canopy,Tree Canopy over Impervious,Agricultural


In [9]:

#difference between years in strings 

import re

#regex to get years from column names
#int(re.search(r'(\d{4})',name).group(1))

int(re.search(r'(\d{4})','canopy_change_2018').group(1)) - int(re.search(r'(\d{4})','canopy_change_2013').group(1))


5